# Multivariate Time Series Forecasting using Stacked LSTM

A cleaned, portfolio-oriented notebook that uses the reusable modules in `src/`.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from src.data_preprocessing import load_time_series_csv, prepare_time_series, chronological_split
from src.feature_engineering import add_calendar_features
from src.sequence_generation import build_supervised_sequences
from src.model_training import build_stacked_lstm, train_model
from src.model_evaluation import regression_metrics, naive_previous_value

## 1. Load and validate the hourly dataset

In [ ]:
data = load_time_series_csv(PROJECT_ROOT / "data" / "hourly_energy.csv")
data, quality_report = prepare_time_series(data)
data = add_calendar_features(data)
quality_report

In [ ]:
data.head(), data.shape, data["timestamp"].min(), data["timestamp"].max()

## 2. Explore target and multivariate relationships

In [ ]:
data.set_index("timestamp")[["energy_load", "temperature", "humidity"]].iloc[:24*14].plot(figsize=(13, 5), subplots=True);
plt.tight_layout();

In [ ]:
data[["energy_load", "temperature", "humidity", "hour", "dayofweek", "weekend", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]].corr().round(3)

## 3. Chronological split and training-only scaling

In [ ]:
FEATURES = ["energy_load", "temperature", "humidity", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "weekend"]
TARGET = "energy_load"
SEQUENCE_LENGTH = 24

train_df, validation_df, test_df = chronological_split(data, 0.70, 0.15)
feature_scaler = StandardScaler().fit(train_df[FEATURES])
target_scaler = StandardScaler().fit(train_df[[TARGET]])
[len(train_df), len(validation_df), len(test_df)]

In [ ]:
def make_partition(frame):
    features = feature_scaler.transform(frame[FEATURES])
    target = target_scaler.transform(frame[[TARGET]]).reshape(-1)
    return build_supervised_sequences(features, target, SEQUENCE_LENGTH, 1)

X_train, y_train = make_partition(train_df)
X_validation, y_validation = make_partition(validation_df)
X_test, y_test = make_partition(test_df)
X_train.shape, y_train.shape, X_validation.shape, X_test.shape

## 4. Build and optionally train the Stacked LSTM

In [ ]:
model = build_stacked_lstm((SEQUENCE_LENGTH, len(FEATURES)))
model.summary()

In [ ]:
# Uncomment to retrain. The repository already contains the supplied pre-trained artifact.
# history = train_model(
#     model, X_train, y_train, X_validation, y_validation,
#     PROJECT_ROOT / "models" / "stacked_lstm_energy.keras", epochs=20, batch_size=64
# )

## 5. Evaluate the saved model on the future test partition

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model(PROJECT_ROOT / "models" / "stacked_lstm_energy.keras", compile=False)
scaled_prediction = model.predict(X_test, verbose=0).reshape(-1)
prediction = target_scaler.inverse_transform(scaled_prediction.reshape(-1, 1)).reshape(-1)
actual = target_scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(-1)
metrics = regression_metrics(actual, prediction)
metrics

In [ ]:
baseline = naive_previous_value(test_df[TARGET].to_numpy(), SEQUENCE_LENGTH)
pd.DataFrame([
    {"model": "Naive previous value", **regression_metrics(actual, baseline)},
    {"model": "Stacked LSTM", **metrics},
])

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(actual[:500], label="Actual")
plt.plot(prediction[:500], label="Stacked LSTM")
plt.title("Actual vs Predicted — First 500 Test Observations")
plt.legend();

## Verified portfolio result

The supplied artifact achieved test MAE **5.028**, RMSE **6.323**, MAPE **5.09%** and R² **0.916** on the chronologically held-out test period.